## *Konteks / Skenario*

Manajemen "Kopi Nusantara" (sekenario yang sama dari Pertemuan 7, 9, dan 10) ingin memiliki alat bantu prediksi omzet harian tiap cabang, agar dapat merencanakan stok bahan baku dengan lebih baik. Anda ditugaskan membangun model regresi untuk memprediksi total_penjualan sebuah pesanan berdasarkan atribut-atributnya.


### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset.

## **SparkSession**

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Pertemuan11-MLlibRegresi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

26/11/19 07:42:36 WARN Utils: Your hostname, albani resolves to a loopback address: 127.0.1.1; using 10.90.67.30 instead (on interface wlp0s20f3)
26/11/19 07:42:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 07:42:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


## **Dataset**

In [2]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


## **Baca CSV dengan Spark**

In [3]:
df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)
print("Total baris:", df.count())

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows

Total baris: 2500


## **A. Feature Engineering**

In [4]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Simpan model indexer (hasil fit) karena dipakai lagi di bagian E
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx").fit(df)
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df)
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx").fit(df)

df_indexed = idx_waktu.transform(idx_cabang.transform(idx_menu.transform(df)))

assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)
df_final = assembler.transform(df_indexed)

df_final.select("nama_menu", "cabang", "waktu_pesan",
                "nama_menu_idx", "cabang_idx", "waktu_pesan_idx").show(5)
df_final.select("features", "total_penjualan").show(5, truncate=False)

+--------------+----------+-----------+-------------+----------+---------------+
|     nama_menu|    cabang|waktu_pesan|nama_menu_idx|cabang_idx|waktu_pesan_idx|
+--------------+----------+-----------+-------------+----------+---------------+
|Kentang Goreng|  Semarang|      Malam|          5.0|       2.0|            1.0|
|Kentang Goreng|Yogyakarta|       Sore|          5.0|       1.0|            3.0|
|Kentang Goreng|  Semarang|      Siang|          5.0|       2.0|            2.0|
|Kentang Goreng|Yogyakarta|       Pagi|          5.0|       1.0|            0.0|
|    Roti Bakar|  Magelang|      Malam|          4.0|       0.0|            1.0|
+--------------+----------+-----------+-------------+----------+---------------+
only showing top 5 rows

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000

## **B. Train/Test Split & Pelatihan Model**

In [5]:
from pyspark.ml.regression import LinearRegression

train, test = df_final.randomSplit([0.75, 0.25], seed=101)
print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing  : {test.count()}")

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model = lr.fit(train)

print("Koefisien:", model.coefficients)
print("Intercept:", model.intercept)

Jumlah data training : 1839
Jumlah data testing  : 661
Koefisien: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66903.17766485889


## **C. Evaluasi Model** 

In [6]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test)

rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction",
                           metricName="rmse").evaluate(predictions)
r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction",
                         metricName="r2").evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% variasi data berhasil dijelaskan model)")

RMSE : Rp 10,901
R²   : 0.9107  (91.1% variasi data berhasil dijelaskan model)


## Penjelasan
R² sekitar 0,92 berarti model menjelaskan sekitar 92% variasi total_penjualan, dengan galat rata-rata (RMSE) sekitar Rp 10.900 pada pesanan yang rata-rata bernilai sekitar Rp 65.000. Model cukup baik untuk estimasi kasar omzet dalam perencanaan stok, tetapi kurang tepat untuk menebak nilai satu pesanan karena galatnya masih sekitar 17% dari rata-rata penjualan.

## **D. Eksperimen Perbandingan Fitur** 

In [7]:
# Model kedua: hanya qty dan harga_satuan
assembler_2 = VectorAssembler(inputCols=["qty", "harga_satuan"], outputCol="features_2")
df_2 = assembler_2.transform(df_indexed)

# Split identik (75/25, seed=101) supaya perbandingannya adil
train_2, test_2 = df_2.randomSplit([0.75, 0.25], seed=101)

model_2 = LinearRegression(featuresCol="features_2", labelCol="total_penjualan").fit(train_2)
pred_2 = model_2.transform(test_2)

r2_2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction",
                           metricName="r2").evaluate(pred_2)

print(f"R² model utama (5 fitur) : {r2:.4f}")
print(f"R² model kedua (2 fitur) : {r2_2:.4f}")
print(f"Selisih                  : {r2_2 - r2:+.4f}")

R² model utama (5 fitur) : 0.9107
R² model kedua (2 fitur) : 0.9108
Selisih                  : +0.0001


## Kesimpulan
Menambahkan informasi menu, cabang, dan waktu pesan tidak meningkatkan akurasi secara berarti. R² model 5 fitur dan 2 fitur nyaris sama (selisih sekitar 0,001, bahkan model 2 fitur sedikit lebih tinggi). Hal ini terjadi karena total_penjualan pada dataset ini dibentuk dari qty × harga_satuan ditambah derau, sehingga menu, cabang, dan waktu pesan tidak punya pengaruh nyata. Selain itu, indeks hasil StringIndexer diperlakukan model linear sebagai angka berurutan padahal kategorinya tidak berurutan.

## **E. Prediksi Skenario Baru**

In [8]:
# 3 skenario pesanan (data dummy)
skenario = spark.createDataFrame([
    ("Kopi Susu",  "Magelang",   "Pagi",  2, 18000),
    ("Cappuccino", "Yogyakarta", "Sore",  3, 25000),
    ("Es Teh",     "Semarang",   "Malam", 5, 15000),
], ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"])

# Pakai StringIndexerModel YANG SAMA dari bagian A (tanpa fit() ulang)
skenario_idx = idx_waktu.transform(idx_cabang.transform(idx_menu.transform(skenario)))
skenario_final = assembler.transform(skenario_idx)

hasil = model.transform(skenario_final)
hasil.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction") \
     .show(truncate=False)

+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Magelang  |Pagi       |2  |18000       |31479.510248098144|
|Cappuccino|Yogyakarta|Sore       |3  |25000       |75861.60625084834 |
|Es Teh    |Semarang  |Malam      |5  |15000       |89202.51678927698 |
+----------+----------+-----------+---+------------+------------------+



## Menutup SparkSession

In [9]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
